# Demo — Asesor de viajes multi-agente como API de producción (Pre-entrega 7)

Este notebook muestra el **orquestador de la PE6 interactuando en vivo a través de la API** (mejora pedida por el corrector):

1. el grafo de producción, generado desde el código, con el nodo human-in-the-loop `aprobacion_web`;
2. un job real: `POST /tasks` → polling de `GET /tasks/{id}` → **pausa** antes de buscar en la web → `POST /approve` con `{"aprobado": true}` → `DONE`;
3. otro job con `{"aprobado": false}`: sigue sin web y la síntesis lo aclara;
4. un job cuya pregunta ya responde la memoria (ChromaDB): **no hay pausa**;
5. la prueba de carga de 5 pedidos concurrentes;
6. apéndice: el flujo de delegación de la PE6 en modo mock, en proceso (sin API ni claves).

**Requisito:** la API corriendo (`docker compose up` o `uvicorn app.main:app --port 8000`). Las salidas guardadas son de la corrida real del 18/11/2026 (Claude `claude-haiku-5-5` + Tavily + Redis Cloud + LangSmith).

In [1]:
import asyncio
import json
import logging
import time
import warnings
from datetime import date

import httpx
from IPython.display import Markdown, display

import config

warnings.filterwarnings("ignore")
config.configurar_logs(logging.WARNING)
API = "http://localhost:8000"
cliente = httpx.AsyncClient(base_url=API, timeout=30)
salud = (await cliente.get("/health")).json()
print(json.dumps(salud, ensure_ascii=False, indent=1))

{
 "status": "ok",
 "modo": "real",
 "modelo": "claude-haiku-5-5",
 "redis": {
  "ok": true,
  "tipo": "redis"
 },
 "checkpointer": "AsyncRedisSaver",
 "langsmith": {
  "tracing": true,
  "proyecto": "pre-entrega-7",
  "clave": true
 },
 "jobs_activos": 0,
 "max_concurrentes": 5
}


## 1. El grafo de producción (Mermaid generado desde el código)

Es el grafo jerárquico de la PE6 (supervisor con `Literal`, rúbrica R1–R6, frenos anti-bucle) con un nodo más: `aprobacion_web`, en el camino supervisor → investigador. Lo mismo imprime `python main.py --mermaid`.

In [2]:
from app.graph import mermaid_produccion

print(mermaid_produccion())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	investigador(investigador)
	analista(analista)
	aprobacion_web(aprobacion_web)
	sintesis(sintesis)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analista --> supervisor;
	aprobacion_web --> investigador;
	investigador --> supervisor;
	supervisor -.-> analista;
	supervisor -. &nbsp;investigador&nbsp; .-> aprobacion_web;
	supervisor -.-> sintesis;
	sintesis --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. Un job en vivo: POST → polling → pausa HITL → approve → DONE

`POST /tasks` vuelve **al instante** con el `job_id` (202): el grafo corre en una tarea de asyncio de fondo. El cliente consulta `GET /tasks/{id}` hasta que el estado cambia.

In [3]:
async def polling(job_id, hasta=("WAITING_APPROVAL", "DONE", "FAILED"), cada=2.0):
    """Como haría un cliente: consulta el estado hasta llegar a uno de `hasta`, mostrando cada cambio."""
    t0, ultimo = time.perf_counter(), None
    while True:
        job = (await cliente.get(f"/tasks/{job_id}")).json()
        marca = (job["estado"], job.get("nodo_actual"))
        if marca != ultimo:
            print(f"{time.perf_counter() - t0:6.1f} s · {job['estado']:<16} nodo: {job.get('nodo_actual') or '-'}")
            ultimo = marca
        if job["estado"] in hasta:
            return job
        await asyncio.sleep(cada)


def resumen(job):
    r = job.get("resultado") or {}
    m = r.get("metricas") or {}
    print(f"estado: {job['estado']} | tramos: {job['tramos']} | duración: {job['duracion_s']} s "
          f"(grafo: {job['tiempo_ejecucion_s']} s) | tokens: {job['total_tokens']:,} | costo: US$ {job['costo_usd']}")
    if r:
        print("ruta:", r["ruta"])
        print(f"permiso web: {r['permiso_web']} | búsquedas web: {m['busquedas_web']} | "
              f"fuentes: {m['fuentes_memoria']} de memoria, {m['fuentes_web']} de la web | ¿entra?: {m['presupuesto_entra']}")
    if job.get("error"):
        print("ERROR:", job["error"])


PREGUNTA_1 = ("Quiero ir 5 días a Mar del Plata en febrero, somos 2 personas, salimos desde Córdoba y tenemos "
              "700.000 pesos. ¿Me armás el viaje y me decís si nos alcanza?")
t0 = time.perf_counter()
r = await cliente.post("/tasks", json={"pregunta": PREGUNTA_1})
print(f"POST /tasks → {r.status_code} en {(time.perf_counter() - t0) * 1000:.0f} ms:", r.json())
job_1 = r.json()["job_id"]
pausado = await polling(job_1)

POST /tasks → 202 en 166 ms: {'job_id': '3e7a28275085484eb18e1732200cd69d', 'estado': 'PENDING', 'links': {'estado': '/tasks/3e7a28275085484eb18e1732200cd69d', 'aprobar': '/tasks/3e7a28275085484eb18e1732200cd69d/approve'}}


   0.2 s · PENDING          nodo: -


   2.3 s · RUNNING          nodo: -


   6.7 s · RUNNING          nodo: supervisor


   8.9 s · WAITING_APPROVAL nodo: aprobacion_web


El grafo se detuvo en `aprobacion_web` (`interrupt()`): el checkpoint quedó en Redis (`AsyncRedisSaver`, thread_id = job_id) y `GET /tasks/{id}` muestra **qué quiere buscar** el investigador en la web y qué temas ya responde la memoria:

In [4]:
print(json.dumps(pausado["aprobacion_pendiente"], ensure_ascii=False, indent=1))

{
 "tipo": "aprobacion_busqueda_web",
 "ronda": 1,
 "destino": "Mar del Plata",
 "temas_sin_memoria": [
  "pasajes",
  "alojamiento",
  "comida",
  "actividades",
  "clima",
  "cambio"
 ],
 "temas_en_memoria": [],
 "consultas_previstas": [
  "precio de pasajes Córdoba → Mar del Plata",
  "precio de alojamiento por noche en Mar del Plata",
  "cuánto cuesta comer en Mar del Plata",
  "actividades y excursiones en Mar del Plata",
  "clima en Mar del Plata en febrero",
  "cotización del dólar oficial, tarjeta y MEP"
 ],
 "instruccion_del_supervisor": "Buscá datos actuales con fuente (URL o memoria mem-...) para un viaje de Córdoba a Mar del Plata, ida 2027-02-10 y vuelta 2027-02-14 (5 días, 4 noches), 2 personas. Necesitamos: (1) pasajes Córdoba-Mar del Plata en ómnibus y/o avión, con precios de referencia por persona (rangos, valores \"desde\" o promedios de fechas cercanas, no hace falta el precio exacto de esas fechas); (2) alojamiento en Mar del Plata para 2 personas por 4 noches en fe

Una persona aprueba. El worker reanuda el grafo desde el checkpoint con `Command(resume={"aprobado": true})`:

In [5]:
r = await cliente.post(f"/tasks/{job_1}/approve", json={"aprobado": True, "comentario": "ok desde el notebook"})
print(r.status_code, r.json())
final_1 = await polling(job_1, hasta=("DONE", "FAILED"))
print()
resumen(final_1)

202 {'job_id': '3e7a28275085484eb18e1732200cd69d', 'estado': 'RUNNING', 'aprobado': True}
   0.2 s · RUNNING          nodo: aprobacion_web


  28.5 s · RUNNING          nodo: investigador


  32.8 s · RUNNING          nodo: supervisor


  50.2 s · RUNNING          nodo: analista


  52.4 s · RUNNING          nodo: supervisor


  63.2 s · RUNNING          nodo: sintesis


  65.4 s · DONE             nodo: -

estado: DONE | tramos: 2 | duración: 74.85 s (grafo: 69.71 s) | tokens: 83,767 | costo: US$ 0.013354
ruta: supervisor → aprobacion_web → investigador → supervisor → analista → supervisor → sintesis
permiso web: aprobada | búsquedas web: 6 | fuentes: 0 de memoria, 18 de la web | ¿entra?: True


In [6]:
display(Markdown(final_1["resultado"]["respuesta_final"]))

## Respuesta corta

Sí, con la opción económica el viaje entra en los $700.000, pero el margen tiene que cubrir comida y actividades, que no tienen precio confiable. Con alojamiento en hotel no alcanza.

## Itinerario

Fechas asumidas: ida el 10/02/2027 y vuelta el 14/02/2027 (febrero de 2026 ya pasó).

- **Día 1 (10/02): llegada y primer paseo.** Llegada en micro desde Córdoba, check-in y paseo por la costa.
- **Día 2 (11/02): bodega.** Visita y degustación en Bodega Trapiche, que figura en Tangol pero sin precio [https://www.tangol.com/esp/argentina/mar-del-plata/excursiones/d_21_tp_3].
- **Día 3 (12/02): centro y audioguías.** Recorrido con las audioguías gratuitas de la oficina de turismo [https://www.turismomardelplata.gob.ar].
- **Día 4 (13/02): día libre.** Playa o descanso. No hay más actividades con datos confirmados.
- **Día 5 (14/02): regreso.**

**Clima esperado:** febrero suele ser cálido y seco. Una nota de un año puntual reporta máximas de 37–38 °C y poca lluvia [https://www.lu9mardelplata.com.ar/nota-febrero-arranca-con-temperaturas-record-en-mar-del-plata-jornada-muy-calurosa-y-escasas-precipitaciones-15823]. Es un caso aislado, no un promedio. La temporada templada va del 7 de diciembre al 20 de marzo, con máximas promedio superiores a 23 °C [https://es.weatherspark.com/y/29078/Tiempo-promedio-en-Mar-del-Plata-Argentina-durante-todo-el-a%C3%B1o].

## Presupuesto

**Opción económica (micro + alojamiento "desde" de Despegar)**

| Renglón | Monto |
|---|---|
| Pasajes micro: ARS 92.130 por persona, ida y vuelta, × 2 | $184.260 |
| Alojamiento: ARS 46.917 por persona por noche × 4 noches × 2 | $375.336 |
| Comida | $0 (sin precio confiable) |
| Actividades | $0 (sin precio confiable) |
| **Total** | **$559.596** |

- **Resultado:** entra en $700.000 y sobran **$140.404**.
- **Costo por día:** $111.919 ($55.960 por persona por día).
- **Pasajes:** [https://www.plataforma10.com.ar/pasajes-en-micro/cordoba/mar-del-plata]
- **Alojamiento:** [https://www.despegar.com.ar/hoteles/hl/4445/i1/hoteles-en-mar+del+plata]

**Tipo de cambio:** no se usó. Todos los precios están en ARS. Las cotizaciones encontradas (oficial, tarjeta y MEP) no tienen una fecha verificable, así que no se aplicaron [https://www.cronista.com/MercadosOnline/dolar.html] [https://www.lanacion.com.ar/dolar-hoy].

**Unidades dudosas:**
- **Pasajes:** la fuente no indica si el precio es por persona o total. Tomamos por persona, ida y vuelta. Si fuera solo ida, el total de pasajes se duplica y el margen se achica mucho.
- **Alojamiento:** tomamos el caso más caro, por persona por noche. Si fuera por habitación, el costo baja a la mitad.

## Alternativas

**Hotel Oscar Lescano:** la oferta de $246.954 es por noche por habitación para 2 personas. Con 4 noches el total sería $1.172.076, es decir, **faltan $472.076**. No conviene con este presupuesto [https://www.despegar.com.ar/hoteles/hl/4445/i1/hoteles-en-mar+del+plata].

Para ajustar más, se puede alquilar un departamento o compartir alojamiento, cocinar parte de las comidas y priorizar actividades gratuitas.

## Lo que no pudimos averiguar

- **Comida:** no hay precios en ARS confirmados. Las fuentes son videos con precios de marzo 2025.
- **Actividades:** sin montos en ARS.
- **Avión:** no se encontraron precios.
- **Pasajes:** no se confirmó la unidad del precio de plataforma10 ni el tramo del precio de $129.000 de centraldepasajes.
- **Alojamiento:** no hay un precio confirmado por noche en ARS para 2 personas.
- **Clima:** no hay un promedio detallado de febrero.

Como comida y actividades están en cero, el costo real va a ser mayor que el calculado. Conviene verificar el precio del alojamiento y los pasajes antes de reservar.

## Fuentes
- [web] https://www.plataforma10.com.ar/pasajes-en-micro/cordoba/mar-del-plata — pasajes, buscado el 2026-11-18
- [web] https://www.despegar.com.ar/hoteles/hl/4445/i1/hoteles-en-mar+del+plata — alojamiento, buscado el 2026-11-18
- [web] https://www.tangol.com/esp/argentina/mar-del-plata/excursiones/d_21_tp_3 — actividades, buscado el 2026-11-18
- [web] https://www.turismomardelplata.gob.ar — actividades, buscado el 2026-11-18
- [web] https://www.lu9mardelplata.com.ar/nota-febrero-arranca-con-temperaturas-record-en-mar-del-plata-jornada-muy-calurosa-y-escasas-precipitaciones-15823 — clima, buscado el 2026-11-18
- [web] https://es.weatherspark.com/y/29078/Tiempo-promedio-en-Mar-del-Plata-Argentina-durante-todo-el-a%C3%B1o — clima, buscado el 2026-11-18
- [web] https://www.lanacion.com.ar/dolar-hoy — cambio, buscado el 2026-11-18
- [web] https://www.cronista.com/MercadosOnline/dolar.html — cambio, buscado el 2026-11-18

_Los precios son aproximados: salen de búsquedas web (o de la memoria local de búsquedas de los últimos días) y pueden cambiar. Confirmalos antes de reservar._

## 3. Rechazar la búsqueda web: `{"aprobado": false}`

El viaje no se corta: el investigador sigue **sin web** (`buscar_en_web` devuelve lo que haya en memoria o un aviso, nunca llama a Tavily) y la síntesis avisa que la respuesta puede estar incompleta.

In [7]:
PREGUNTA_2 = "¿Qué clima hace en El Calafate en mayo y qué excursiones hay?"
job_2 = (await cliente.post("/tasks", json={"pregunta": PREGUNTA_2})).json()["job_id"]
pausado_2 = await polling(job_2)
print("quiere buscar:", pausado_2["aprobacion_pendiente"]["consultas_previstas"])
r = await cliente.post(f"/tasks/{job_2}/approve", json={"aprobado": False, "comentario": "sin gastar búsquedas"})
print(r.status_code, r.json())
final_2 = await polling(job_2, hasta=("DONE", "FAILED"))
print()
resumen(final_2)
display(Markdown(final_2["resultado"]["respuesta_final"]))

   0.2 s · PENDING          nodo: -


   2.3 s · RUNNING          nodo: -


   4.5 s · RUNNING          nodo: supervisor


   8.9 s · WAITING_APPROVAL nodo: aprobacion_web
quiere buscar: ['clima en El Calafate en las fechas del viaje', 'actividades y excursiones en El Calafate']


202 {'job_id': '1dbd949ab22848b5ae993db89954c958', 'estado': 'RUNNING', 'aprobado': False}
   0.2 s · RUNNING          nodo: aprobacion_web


  11.0 s · RUNNING          nodo: investigador


... (2 outputs left out)

  24.1 s · RUNNING          nodo: supervisor


  26.3 s · RUNNING          nodo: sintesis


  28.5 s · DONE             nodo: -

estado: DONE | tramos: 2 | duración: 37.3 s (grafo: 31.39 s) | tokens: 36,952 | costo: US$ 0.005331
ruta: supervisor → aprobacion_web → investigador → supervisor → aprobacion_web → investigador → supervisor → sintesis
permiso web: rechazada | búsquedas web: 0 | fuentes: 0 de memoria, 0 de la web | ¿entra?: None


> ⚠️ La búsqueda web no fue autorizada: esta respuesta usa solo los datos que ya estaban en la memoria local de búsquedas, así que puede estar incompleta.

## Lo que no pudimos averiguar

No pudimos conseguir datos sobre El Calafate en mayo. La búsqueda web no fue autorizada por la persona que revisa el pedido, y la memoria local no tiene información sobre el tema, así que no hay fuentes disponibles para responder.

**Clima en mayo:** no tenemos temperaturas medias, lluvias, viento ni recomendaciones de abrigo.

**Excursiones y actividades:** no tenemos información sobre el Glaciar Perito Moreno, la navegación por el Lago Argentino, el trekking ni las excursiones a El Chaltén. Tampoco tenemos precios de referencia ni horarios de operación en mayo.

## Cómo seguir

Para responder tu consulta hace falta que el revisor autorice la búsqueda web sobre clima y actividades en El Calafate, o que se cargue en la memoria información confiable sobre estos temas.

Si nos decís el origen, las fechas y la cantidad de días, también podemos armar un presupuesto de viaje una vez que tengamos esos datos.

## Fuentes
- No se usaron fuentes externas.

Un segundo `approve` sobre un job que ya no está esperando devuelve **409** (la transición WAITING_APPROVAL → RUNNING es atómica en Redis con WATCH/MULTI):

In [8]:
r = await cliente.post(f"/tasks/{job_2}/approve", json={"aprobado": True})
print(r.status_code, r.json())

409 {'detail': 'El job 1dbd949ab22848b5ae993db89954c958 no está esperando aprobación (estado: DONE).'}


## 4. Si la memoria responde, no hay pausa

Se repite la pregunta del punto 2. Todo lo que el investigador trajo de la web quedó en ChromaDB, así que el nodo `aprobacion_web` ve datos vigentes de cada tema y **no detiene el grafo** (permiso `sin_pedir`).

In [9]:
job_3 = (await cliente.post("/tasks", json={"pregunta": PREGUNTA_1})).json()["job_id"]
final_3 = await polling(job_3)
print()
resumen(final_3)
print("aprobaciones HITL:", [(a["resultado"], a["origen"], a["pausa"]) for a in final_3["resultado"]["aprobaciones_hitl"]])

   0.2 s · PENDING          nodo: -


   2.3 s · RUNNING          nodo: -


   6.7 s · RUNNING          nodo: aprobacion_web


  28.4 s · RUNNING          nodo: investigador


  32.8 s · RUNNING          nodo: supervisor


  61.0 s · RUNNING          nodo: analista


  63.2 s · RUNNING          nodo: supervisor


  76.2 s · RUNNING          nodo: sintesis


  78.4 s · DONE             nodo: -

estado: DONE | tramos: 1 | duración: 76.43 s (grafo: 74.13 s) | tokens: 117,141 | costo: US$ 0.017909
ruta: supervisor → aprobacion_web → investigador → supervisor → analista → supervisor → sintesis
permiso web: sin_pedir | búsquedas web: 0 | fuentes: 15 de memoria, 0 de la web | ¿entra?: False
aprobaciones HITL: [('sin_pedir', 'memoria', False)]


## 5. Los jobs, uno al lado del otro, y la prueba de carga

Cada corrida también queda en LangSmith (proyecto `pre-entrega-7`, metadata `job_id`): las capturas están en `screenshots/`.

In [10]:
filas = []
for nombre, j in (("aprobado", final_1), ("rechazado", final_2), ("memoria", final_3)):
    m = j["resultado"]["metricas"]
    filas.append(f"| {nombre} | `{j['job_id'][:8]}` | {j['estado']} | {j['tramos']} | {j['duracion_s']} | "
                 f"{j['total_tokens']:,} | {j['costo_usd']} | {m['busquedas_web']} | {j['resultado']['permiso_web']} |")
display(Markdown("| caso | job | estado | tramos | duración (s) | tokens | costo US$ | búsquedas web | permiso web |\n"
                 "|---|---|---|---:|---:|---:|---:|---:|---|\n" + "\n".join(filas)))

| caso | job | estado | tramos | duración (s) | tokens | costo US$ | búsquedas web | permiso web |
|---|---|---|---:|---:|---:|---:|---:|---|
| aprobado | `3e7a2827` | DONE | 2 | 74.85 | 83,767 | 0.013354 | 6 | aprobada |
| rechazado | `1dbd949a` | DONE | 2 | 37.3 | 36,952 | 0.005331 | 0 | rechazada |
| memoria | `356a6b63` | DONE | 1 | 76.43 | 117,141 | 0.017909 | 0 | sin_pedir |

In [11]:
from pathlib import Path

display(Markdown(Path("screenshots/resultados_carga.md").read_text(encoding="utf-8")))

# Prueba de carga — 5 pedidos concurrentes (18/11/2026 14:40)

API en modo **real** · redis · AsyncRedisSaver · modelo `claude-haiku-5-5` · LangSmith proyecto `pre-entrega-7` · aprobación: `script`

| job_id | pregunta | estado | duración (s) | ejecución (s) | tokens | costo estimado | costo LangSmith | búsquedas web | aprobaciones HITL |
|---|---|---|---:|---:|---:|---:|---:|---:|---|
| `17d521cf` | Quiero ir 7 días a Bariloche en enero, somos 2 personas… | DONE | 79.3 | 73.1 | 111,905 | $0.0164 | $0.0164 | 1 | aprobada (persona, pausa) |
| `abf031f5` | Queremos ir 5 días a Salta en abril, somos 3 personas, … | DONE | 85.7 | 82.6 | 105,389 | $0.0165 | $0.0165 | 0 | sin_pedir (memoria), sin_pedir (memoria) |
| `9c06a718` | Somos 2 y queremos ir 4 días a Mendoza en marzo desde B… | DONE | 92.3 | 89.2 | 143,798 | $0.0214 | $0.0214 | 0 | sin_pedir (memoria), sin_pedir (memoria) |
| `1ec8b191` | ¿Qué clima hace en Ushuaia en agosto y qué actividades … | DONE | 35.3 | 32.4 | 29,096 | $0.0048 | $0.0048 | 0 | sin_pedir (memoria) |
| `f996b969` | Quiero ir 6 días a Puerto Madryn en octubre, somos 2 pe… | DONE | 70.4 | 62.6 | 71,031 | $0.0117 | $0.0117 | 0 | aprobada (persona, pausa) |

- **DONE:** 5/5 · **tiempo total:** 92.8 s
- **Latencia por job** (de POST a DONE, incluye la espera de aprobación): p50 **79.3 s** · p95 **91.0 s**
- **Costo total estimado:** $0.0708 (LangSmith: $0.0708)
- Detalle completo (JSON): `traces/carga/carga_real_20261118-144057.json`


---
## Apéndice: el flujo de delegación de la PE6, en proceso y en modo mock

Sin API, sin claves: el mismo grafo con el LLM simulado y Tavily falso (como en la PE6). Sirve para ver el detalle de cada decisión del supervisor y de cada especialista.

In [12]:
from graph import construir_grafo, consultar, crear_componentes

DEMO = ("Quiero ir 7 días a Bariloche en enero, somos 2 personas, salimos desde Mendoza y tenemos "
        "800.000 pesos. ¿Me armás el viaje y me decís si nos alcanza?")
componentes_mock = crear_componentes(modo_mock=True)
grafo_mock = construir_grafo(componentes_mock)

In [13]:
def mostrar_recorrido(traza):
    print("RUTA:", traza.texto_ruta())
    print()
    for e in traza.eventos:
        d = e.detalle
        if e.nodo == "supervisor":
            if "pedido_extraido" in d:
                print("SUPERVISOR · pedido extraído:", d["pedido_extraido"])
            freno = "  (FORZADA POR EL FRENO)" if d.get("forzada") else ""
            print(f"SUPERVISOR · decisión {d['paso']} → {d['siguiente']}{freno}")
            print("   evaluación (rúbrica):", d["evaluacion"][:400])
            if d.get("temas"):
                print("   temas pedidos:", ", ".join(d["temas"]))
        elif e.nodo in ("investigador", "analista"):
            print(f"{e.nodo.upper()} · herramientas: {', '.join(d['herramientas']) or 'ninguna'} "
                  f"· llamadas: {len(d['llamadas'])}")
            for c in d.get("cobertura", []):
                print(f"   cobertura {c['tema']}: {c['fuentes']} fuentes ({c['memoria']} memoria / {c['web']} web)"
                      f"{' · con precios' if c['con_precios'] else ''}")
            print("   chequeo por código:", "; ".join(d["observaciones"]) or "sin observaciones")
        else:
            print("SÍNTESIS · respuesta final lista")
    m = traza.metricas
    print()
    print(f"Decisiones: {m['pasos_supervisor']} | validación: {m['validacion']} | ¿entra?: {m['presupuesto_entra']} "
          f"| datos: {traza.texto_origenes()} | {traza.etiqueta_busquedas()}: {m['busquedas_web']} | {traza.duracion_s} s")


traza_mock = await consultar(grafo_mock, DEMO, escenario="notebook_demo", modo_mock=True)
mostrar_recorrido(traza_mock)

RUTA: supervisor → investigador → supervisor → analista → supervisor → sintesis

SUPERVISOR · pedido extraído: {'origen': 'Mendoza', 'destino': 'Bariloche', 'fecha_ida': '2027-01-10', 'fecha_vuelta': '2027-01-16', 'dias': 7, 'personas': 2, 'presupuesto': 800000.0, 'moneda': 'ARS', 'supuestos': ['enero sin día exacto: se asume del 10/01/2027 al 16/01/2027']}
SUPERVISOR · decisión 1 → investigador
   evaluación (rúbrica): R1: todavía no hay datos. Empieza el investigador.
   temas pedidos: pasajes, alojamiento, clima, actividades, cambio
INVESTIGADOR · herramientas: buscar_en_memoria, buscar_en_web · llamadas: 10
   cobertura pasajes: 2 fuentes (0 memoria / 2 web) · con precios
   cobertura alojamiento: 1 fuentes (0 memoria / 1 web) · con precios
   cobertura clima: 1 fuentes (0 memoria / 1 web)
   cobertura actividades: 1 fuentes (0 memoria / 1 web) · con precios
   cobertura cambio: 1 fuentes (0 memoria / 1 web) · con precios
   chequeo por código: sin observaciones
SUPERVISOR · decisi

In [14]:
display(Markdown(traza_mock.respuesta_final))

## Itinerario
- Día 1 (10/01): viaje desde Mendoza y llegada a Bariloche
- Día 2 (11/01): Circuito Chico en bici
- Día 3 (12/01): trekking al Refugio Frey
- Día 4 (13/01): playas del lago Gutiérrez
- Día 5 (14/01): excursión a Isla Victoria y Bosque de Arrayanes
- Día 6 (15/01): día libre: descanso y paseo por el centro
- Día 7 (16/01): regreso a Mendoza

Clima esperado: Enero es pleno verano en Bariloche: máximas promedio de 22 °C y mínimas de 7 °C, con días largos (anochece cerca de las 21:30). Llueve poco (unos 4 días al mes), pero hay viento y las noches son frescas: conviene llevar abrigo liviano y protector solar.

## Presupuesto
PRESUPUESTO — opción 'económica' (7 días, 6 noches, 2 personas):
- Pasajes micro semicama: $ 280.000  [$ 140.000 por persona (ida y vuelta) × 2 personas]
- Alojamiento hostel habitación doble: $ 210.000  [$ 35.000 por noche por habitación × 6 noches × 1 habitación (de 2 personas)]
- Comida: $ 126.000  [$ 9.000 por persona por día × 7 días × 2 personas]
- Actividades: $ 56.000  [$ 4.000 por persona por día × 7 días × 2 personas]
- excursión Isla Victoria: $ 113.100  [$ 56.550 × 2 personas]
TOTAL: $ 785.100 | PRESUPUESTO: $ 800.000
RESULTADO: ENTRA en el presupuesto. Sobran $ 14.900.
Costo por día: $ 112.157 | por persona por día: $ 56.079
Conversiones de moneda:
- excursión Isla Victoria: US$ 30,00 → $ 56.550 con dólar tarjeta $1.885 (pago con tarjeta; fuente: https://cotizaciones.mock-viajes.example/dolar-hoy)

**Con la opción económica el viaje ENTRA en el presupuesto.**

## Fuentes
- [web] https://cotizaciones.mock-viajes.example/dolar-hoy — cambio, buscado el 2026-11-18
- [web] https://micros.mock-viajes.example/mendoza-bariloche — pasajes, buscado el 2026-11-18
- [web] https://alojamiento.mock-viajes.example/bariloche-enero-precios — alojamiento, buscado el 2026-11-18
- [web] https://actividades.mock-viajes.example/bariloche-verano — comida, buscado el 2026-11-18

_Los precios son aproximados: salen de búsquedas web (o de la memoria local de búsquedas de los últimos días) y pueden cambiar. Confirmalos antes de reservar._

### A.2 Memoria: la segunda vez no se busca en la web

Se repite la pregunta de demo en mock, con la **misma** memoria ChromaDB de arriba. El investigador consulta primero `buscar_en_memoria`; como los datos tienen menos de 7 días y son parecidos a la consulta, no llama a Tavily.

In [15]:
segunda = await consultar(grafo_mock, DEMO, escenario="notebook_memoria", modo_mock=True)
print("Primera vez →", traza_mock.texto_origenes(), "| búsquedas web:", traza_mock.metricas["busquedas_web"])
print("Segunda vez →", segunda.texto_origenes(), "| búsquedas web:", segunda.metricas["busquedas_web"])

Primera vez → 0 de memoria · 6 de la web | búsquedas web: 5
Segunda vez → 6 de memoria · 0 de la web | búsquedas web: 0


### A.3 El freno de pasos (contra el "supervisor infinito")

Con `max_pasos=2`, la segunda decisión del supervisor es **FINALIZAR forzado por código** (sin consultar al LLM) y la síntesis avisa que la información es parcial.

In [16]:
freno = await consultar(construir_grafo(crear_componentes(modo_mock=True), max_pasos=2), DEMO,
                       escenario="notebook_freno", modo_mock=True, max_pasos=2)
mostrar_recorrido(freno)
print()
print(freno.respuesta_final.splitlines()[0])

RUTA:

 supervisor → investigador → supervisor → sintesis

SUPERVISOR · pedido extraído: {'origen': 'Mendoza', 'destino': 'Bariloche', 'fecha_ida': '2027-01-10', 'fecha_vuelta': '2027-01-16', 'dias': 7, 'personas': 2, 'presupuesto': 800000.0, 'moneda': 'ARS', 'supuestos': ['enero sin día exacto: se asume del 10/01/2027 al 16/01/2027']}
SUPERVISOR · decisión 1 → investigador
   evaluación (rúbrica): R1: todavía no hay datos. Empieza el investigador.
   temas pedidos: pasajes, alojamiento, clima, actividades, cambio
INVESTIGADOR · herramientas: buscar_en_memoria, buscar_en_web · llamadas: 10
   cobertura pasajes: 2 fuentes (0 memoria / 2 web) · con precios
   cobertura alojamiento: 1 fuentes (0 memoria / 1 web) · con precios
   cobertura clima: 1 fuentes (0 memoria / 1 web)
   cobertura actividades: 1 fuentes (0 memoria / 1 web) · con precios
   cobertura cambio: 1 fuentes (0 memoria / 1 web) · con precios
   chequeo por código: sin observaciones
SUPERVISOR · decisión 2 → FINALIZAR  (FORZADA PO

## Dónde está la evidencia

- `screenshots/` → capturas de LangSmith (traza completa, pausa HITL, costo por ejecución, latencia p95) y `resultados_carga.md`.
- `traces/api/` → una traza `.json` + `.log` por job real de la API; `traces/carga/` → el detalle de cada prueba de carga.
- `traces/` → las trazas de los escenarios de la PE6.
- `tests/` → los tests con pytest, sin claves (`python -m pytest -q`).